*openmobisim tutorials* · [00 · overview](00_getting_started.ipynb) · ← [05 · Results and figures](05_results_and_figures.ipynb) · [07 · Validation](07_validation.ipynb) → · [documentation](https://openmobisim.github.io)

###### 06 · Parameters and experiments

Every number openmobisim assumes has a name, a default and a way to change it. None is calibrated: the defaults are
reasonable starting values from the literature or plain assumptions, to be replaced by a study's own. This notebook
lists them, changes some, and runs small experiments: seeds and replications, sampled demand, a sweep, and a city's
mode shares moved by one constant.

---

###### Installation

> Skip this if `import openmobisim` already works in this notebook. **One install is all there is:**
> `pip install "openmobisim[all]"` brings the simulator and everything the ten tutorials use (JupyterLab,
> matplotlib for the figures, pandas, polars and pyarrow for the tables); nothing is installed later. Find the case
> that matches yours and copy its commands, in order. Each needs Python 3.11 or newer.
>
> **A new conda environment** (any system; openmobisim is on PyPI, so pip installs it into the environment):
>
> ```bash
> conda create -y -n openmobisim python=3.12
> conda activate openmobisim
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> **A new venv on macOS or Linux** (in a terminal):
>
> ```bash
> python3 -m venv openmobisim
> source openmobisim/bin/activate
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> **A new venv on Windows** (in the Command Prompt):
>
> ```bat
> py -m venv openmobisim
> openmobisim\Scripts\activate
> pip install "openmobisim[all]"
> openmobisim notebooks
> ```
>
> The last line copies these ten notebooks into `./openmobisim-notebooks` and opens them in JupyterLab. In VS Code or
> PyCharm, open a notebook and choose the `openmobisim` environment's Python as its kernel instead.
>
> **Into the environment this notebook already runs in**: paste this into a new code cell, run it, then restart the
> kernel.
>
> ```python
> %pip install "openmobisim[all]"
> ```

---

**Contents**

1. [Every parameter in one table](#1-·-Every-parameter-in-one-table)
2. [Changing them by name](#2-·-Changing-them-by-name)
3. [Seeds and replications](#3-·-Seeds-and-replications)
4. [Fewer travellers, each standing for more people](#4-·-Fewer-travellers,-each-standing-for-more-people)
5. [A small sweep](#5-·-A-small-sweep)
6. [Mode shares: plausible defaults, yours to change](#6-·-Mode-shares:-plausible-defaults,-yours-to-change)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import openmobisim as ms

net = ms.examples.manhattan_grid(n=6, block_metres=250.0, signals=True)
cars = {"commuter": (True, False, False)}


def demand(count, seed=2):
    return ms.examples.trips_random(net, count, seed=seed, spread_s=3600, min_m=500, max_m=2000)


def mean_trip_min(run):
    return run.mean_travel_time_s / 60  # per person: a traveller may stand for several

###### 1 · Every parameter in one table

`ms.parameters()` lists them all at the defaults a `Scenario` uses: the group, the argument that changes it, its name,
value and unit.

In [ ]:
p = pd.DataFrame(ms.parameters())
print(len(p), "parameters")
p.groupby("group").size().rename("parameters").to_frame()

In [ ]:
p[p["group"].isin(["loading", "modes"])][["argument", "name", "value", "unit", "note"]]

###### 2 · Changing them by name

Each group has its argument: `choice_options`, `loading_options`, `equilibration_options`, `mode_options`,
`transit_options`, `parking_options`, `route_options` of `Scenario.from_parts`, and `network_options` of the network
readers (`network_read_osm`, `network_read_table`, `network_read_gmns`). A name that does not exist is refused with
the list of those that do, so a typo never passes silently.

In [ ]:
try:
    ms.Scenario.from_parts(net, demand(10), classes=cars, choice_options={"beta_tim_min": -0.1}).run("typo")
except ValueError as e:
    print(e)

Here, the value of time: the default logit weighs a minute at −0.2. Halving it makes travellers less sensitive to time,
so more of them take routes a little slower than the best; doubling it, fewer.

In [ ]:
trips = demand(20000)  # enough for queues, so that route choice matters
rows = []
for beta in (-0.1, -0.2, -0.4):
    run = ms.Scenario.from_parts(
        net, trips, classes=cars, choice_options={"beta_time_min": beta}
    ).run(f"beta{beta}", output_dir=f"runs/beta{beta}", quiet=True)
    c = run.convergence()
    rows.append({
        "beta_time_min": beta,
        "mean trip (min)": mean_trip_min(run),
        "gap": c["gap"][-1],
        "expected by the model": c["gap_expected"][-1],
        "fingerprint": run.fingerprint,
    })
pd.DataFrame(rows).set_index("beta_time_min").round(4)

The gap the model itself expects grows as time matters less: those travellers chose slower routes on purpose. Each
setting is a different run, with its own fingerprint.

###### 3 · Seeds and replications

A logit model draws who takes which route. The draws come from one number, `master_seed`: the same seed repeats a run
exactly; another seed is another sample of the same model, and the spread over seeds is the run's sampling noise.

In [ ]:
reps = []
for seed in range(5):
    run = ms.Scenario.from_parts(net, trips, classes=cars, master_seed=seed).run(
        f"seed{seed}", output_dir=f"runs/seed{seed}", quiet=True
    )
    reps.append({"master_seed": seed, "mean trip (min)": mean_trip_min(run)})
reps = pd.DataFrame(reps).set_index("master_seed")
print(reps.round(4))
print(f"mean {reps.iloc[:, 0].mean():.4f} min, standard deviation {reps.iloc[:, 0].std():.4f} min")

###### 4 · Fewer travellers, each standing for more people

`demand_sample(trips, weight)` keeps one traveller in every `weight`, each standing for `weight` people (their vehicle
counts as that many PCU in the loading). The draw keeps each class's and each mode's share to within one traveller.
The run is cheaper, though by less than the weight (what does not grow with demand stays). The mean trip is per
person (`Run.mean_travel_time_s`).

In [ ]:
big = demand(20000)
rows = []
for weight in (1, 2, 4, 8):
    sample = ms.demand_sample(big, weight, seed=0)
    run = ms.Scenario.from_parts(net, sample, classes=cars).run(
        f"weight{weight}", output_dir=f"runs/weight{weight}", quiet=True
    )
    t = run.timings().to_pandas()
    rows.append({
        "weight": weight,
        "travellers simulated": len(sample),
        "mean trip (min)": mean_trip_min(run),
        "run time (s)": float(t.loc[t["stage"] == "total", "seconds"].iloc[0]),
    })
pd.DataFrame(rows).set_index("weight").round(3)

**Under congestion a sample overstates the delays**, and more so the coarser it is: a traveller of weight 8 is one vehicle
of 8 PCU, so its queues form and clear in lumps (a known effect of sampled demand in traffic simulation). On this busy,
signalised grid the mean trip is 63 % too high at weight 8; on a city's network at mild congestion the bias measured a
few per cent at weights 5 to 10. A sample is for screening many scenarios quickly; a result to report comes from the
full demand, or from a weight whose bias has been checked against it, as here.

###### 5 · A small sweep

How the mean trip grows with demand on this grid, at free flow and with queues: a loop over scenarios is all an
experiment needs.

In [ ]:
counts = [2000, 6000, 10000, 14000, 18000, 22000]
free, loaded = [], []
for n in counts:
    trips_n = demand(n)
    free.append(mean_trip_min(ms.Scenario.from_parts(
        net, trips_n, classes=cars, flow_level=0, equilibration="free_flow", choice_model="deterministic"
    ).run(f"free{n}", output_dir=f"runs/free{n}", quiet=True)))
    loaded.append(mean_trip_min(ms.Scenario.from_parts(net, trips_n, classes=cars).run(
        f"loaded{n}", output_dir=f"runs/loaded{n}", quiet=True
    )))

fig, ax = plt.subplots(figsize=(16 / 2.54, 8 / 2.54), dpi=200)
ax.plot(counts, free, marker="o", label="free flow")
ax.plot(counts, loaded, marker="o", label="with queues (default)")
ax.set_xlabel("trips in the hour")
ax.set_ylabel("mean trip (min)")
ax.legend()
ax.set_title("A 6 x 6 grid under growing demand")
plt.show()
pd.DataFrame({"free flow (min)": free, "with queues (min)": loaded}, index=pd.Index(counts, name="trips")).round(2)

###### 6 · Mode shares: plausible defaults, yours to change

The starter cities' traveller classes carry **mode constants** fitted so that each city's residents drive on the
share of their trips its survey gives: Amsterdam's 23 % (residents' trips; they cycle on 38 %), Paris's of trips within Paris
(EMG 2023, https://www.paris.fr/pages/inedit-le-velo-surpasse-la-voiture-a-paris-et-en-petite-couronne-et-de-loin-26832),
Lyon's (Enquête Déplacements 2015,
https://www.urbalyon.org/sites/default/files/productions/publications/imports/2019-11/m-emd-agglo.pdf). They are
**chosen to be logical and to fit those numbers, not calibrated**: a study sets them to its own city and survey.

**Start small: Delft.** The smallest starter city runs its whole morning in about twenty seconds — the split its example classes
give (the case's own travellers and what enters from outside, by people):

In [ ]:
# Delft's whole morning: every departure (06:30-09:30), run on to 10:30 so the late ones arrive.
delft_scenario = ms.examples.case("delft").scenario("base", departures="all", window=("06:30", "10:30"))
delft_run = delft_scenario.run("delft-base", output_dir="runs/delft-base", quiet=True)
delft_modes = delft_run.trip_modes()
delft_people = pd.Series(delft_modes["weight"], index=pd.Index(delft_modes["mode"], name="mode")).groupby(level=0).sum()
print(f"run in {delft_run.timings()['seconds'][-1]:.1f} s")
(100 * delft_people / delft_people.sum()).round(1).rename("Delft (%)")

One constant moves a split. Here, Amsterdam's morning with the bike constant of the two classes that own a bike at its
fitted value (1.23) and one unit either side, on the case's fast run (each simulated traveller five people), counting
the city's own travellers (the survey's), not what enters from outside. A secant step or two on
such a constant fits any target; a car or transit constant works the same way.

In [ ]:
import collections
import copy

case = ms.examples.case("amsterdam")
city = case.network()
timetable = case.transit(city)
parks = case.parkings()
example = case.classes()  # the package's example classes, and those of what enters from outside (notebook 02)
sample = case.demand()  # the fast run: five people a simulated traveller
outside = {"from_outside_car", "from_outside_transit", "goods"}  # fixed modes: not the survey's travellers
own = {t[0] for t in sample if t[7] not in outside}


def with_bike_constant(value):
    """The example classes, the bike constant of those that own a bike set to ``value``."""
    classes = copy.deepcopy(example)
    for c in classes.values():
        if "beta_mode_bike" in c["betas"]:
            c["betas"]["beta_mode_bike"] = value
    return classes


rows = []
for value in (0.23, 1.23, 2.23):
    run = ms.Scenario.from_parts(
        city, sample, classes=with_bike_constant(value), transit=timetable, parkings=parks,
        **case.info["scenarios"]["base"],  # the case's settings: shared searches, stop at a good gap
    ).run(f"bike{value}", output_dir=f"runs/bike{value}", quiet=True)  # fmt: skip
    modes = run.trip_modes()
    people = collections.Counter()
    for who, mode, weight in zip(modes["traveller_id"], modes["mode"], modes["weight"]):
        if who in own:
            people[mode] += weight
    total = sum(people.values())
    rows.append({
        "bike constant": value,
        "bike (%)": 100 * people["bike"] / total,
        "car (%)": 100 * people["car"] / total,
        "transit (%)": 100 * people["transit"] / total,
        "bike : car": people["bike"] / people["car"],
    })
pd.DataFrame(rows).set_index("bike constant").round(2)

A unit of constant is five minutes of riding at the default time weight (−0.2 per minute), so the fitted 1.23 says that
Amsterdam's cyclists prefer the bike by about six minutes over what times alone would give. It was fitted on this
morning so that the city's residents drive on 23 % of their trips, the survey's share (the middle row), and the table
shows how far a unit of constant moves them. The bike then takes more than the survey's 38 %, and walking much less
than a real morning has: the synthetic demand lacks most of the short trips people walk, a known limit of its recipe.
Fit the constants on the demand you will run.

**Next:** [07 · Validation](07_validation.ipynb): validation: the mechanisms against hand calculations, queueing theory and the benchmarks. The whole series: [00 · overview](00_getting_started.ipynb). The documentation: [openmobisim.github.io](https://openmobisim.github.io). Something wrong or unclear? [Report it](https://github.com/bahmanmdd/openmobisim/issues).